# 04 — Final E2-GP classification model

Compare three representations using reproducible holdout splits, then evaluate
the uncalibrated forest by five-fold cross-validation. The operational classifier
is a class-balanced Random Forest with 150 trees, minimum leaf size three and
three-fold sigmoid calibration. Export that calibrated model and its ordered
304-feature contract for external evaluation and subsequent active learning.

Keep the holdout comparison, base-forest CV and calibrated OOF evaluation
separate: they use different estimator or evaluation specifications.


## 0. Configuration and feature-loading utilities


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, tempfile
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'mpl_cache'))

ROOT = Path.cwd()
if not (ROOT / 'config.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from src.features import statistics as est
from src.models.evaluation import evaluate, cross_validated_predictions
from src.models.classification import CLASS_NAMES

RS = config.RANDOM_STATE
N_ESTIMATORS = 150


_DF  = pd.read_csv(config.FEATURES_CSV)       
_GP1 = pd.read_csv(config.FEATURES_GP_CSV)    
_GP2 = pd.read_csv(config.FEATURES_GP2D_CSV)  


def features(variant_name):
    'Features.'
    if variant_name == 'baseline':
        d = _DF
        cols = [c for c in d.columns if c not in ('object_id', 'target')]
    elif variant_name == 'gp1':
        d = est.add_derived_features(_DF.merge(_GP1, on='object_id'))
        cols = [c for c in d.columns if c not in ('object_id', 'target')]
    elif variant_name == 'gp1gp2':
        d = est.add_derived_features(
            _DF.merge(_GP1, on='object_id').merge(_GP2, on='object_id'))
        cols = [c for c in d.columns if c not in ('object_id', 'target')]
    else:
        raise ValueError(f"variante unknown: {variant_name!r}")
    cols = list(dict.fromkeys(cols))  
    return d[cols].values, d['target'].values, cols


def matriz_confusion_global(y_true, y_pred, plot_title, save=None):
    'Matriz confusion global.'
    class_codes = sorted(np.unique(y_true))
    display_names = [CLASS_NAMES.get(c, str(c)) for c in class_codes]
    cm = confusion_matrix(y_true, y_pred, labels=class_codes, normalize='true')
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(class_codes))); ax.set_xticklabels(display_names, rotation=90)
    ax.set_yticks(range(len(class_codes))); ax.set_yticklabels(display_names)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class'); ax.set_title(plot_title)
    for i in range(len(class_codes)):
        for j in range(len(class_codes)):
            if cm[i, j] >= 0.01:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha='center', va='center',
                        color='white' if cm[i, j] > 0.5 else 'black', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    if save is not None:
        fig.savefig(save, dpi=150)
    plt.show()
    return class_codes, cm


## 1. Holdout comparison of three representations


In [ ]:
VARIANTS = [
    ('RF baseline',              'baseline', False),
    ('RF + stats + GP1D',        'gp1',      True),
    ('E2-GP · stats+GP1D+GP2D',  'gp1gp2',   True),
]

result_records, models = [], {}
for item_name, pair_key, balance in VARIANTS:
    X, y, cols = features(pair_key)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=RS, stratify=y)
    rf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS, class_weight='balanced' if balance else None,
        random_state=RS, n_jobs=-1)
    estimator_model, _ = evaluate(f"{item_name} [{len(cols)}f]", rf,
                        X_tr, y_tr, X_te, y_te, result_records, pair_key)
    models[pair_key] = estimator_model


In [ ]:
result_table = pd.DataFrame(result_records).sort_values('MF1', ascending=False).reset_index(drop=True)
result_table.to_csv(config.DIR_PROCESSED / 'resultados_clasificacion_150.csv', index=False)
display(result_table[['variante', 'MF1', 'WF1', 'accuracy', 'notas']])


## 2. Five-fold cross-validation of the uncalibrated forest

Report the mean and standard deviation across folds, the aggregated OOF
classification report and the confusion matrix. Aggregated OOF macro-F1 need
not equal the mean fold macro-F1.


In [ ]:
PRED_CV = {}  

for pair_key, plot_title in [('gp1gp2', 'E2-GP · stats+GP1D+GP2D')]:
    X, y, cols = features(pair_key)
    rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, class_weight='balanced',
                                random_state=RS, n_jobs=-1)
    mf1 = cross_val_score(rf, X, y,
            cv=StratifiedKFold(5, shuffle=True, random_state=RS),
            scoring='f1_macro', n_jobs=-1)
    y_pred = cross_validated_predictions(rf, X, y, n_folds=5, seed_value=RS)

    print('=' * 72)
    print(f"{plot_title}   ({len(cols)} features)")
    print(f"MF1 CV (5 folds): {mf1.mean():.4f} ± {mf1.std():.4f}")
    print(classification_report(y, y_pred, zero_division=0))

    class_codes, cm = matriz_confusion_global(
        y, y_pred,
        f"Aggregated OOF confusion matrix · five-fold CV · {plot_title} ({len(cols)}f)",
        save=config.FIGURE_DIR / f"cm_04_{pair_key}.png")
    PRED_CV[pair_key] = (y, y_pred, cm, class_codes)


## 3. Export the operational classifier and evaluate batch 4

Fit the calibrated classifier on the entire labelled training set. Serialize it
with software versions, feature order and checksums. Evaluate known classes
in the canonical batch-4 subsample; retain 99x objects only for novelty evaluation.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from src.models import classification

tr_def = classification.features_e2gp('train')
X_tr_def, cols_def = classification.e2gp_matrix(tr_def)
y_tr_def = tr_def['target'].to_numpy()

te_def = classification.features_e2gp('test', batch=4)
X_te_def, _ = classification.e2gp_matrix(te_def, cols_def)
rutas_b4 = config.BatchConfig(4)
meta_b4 = pd.read_csv(rutas_b4.test_metadata_clean_csv)[['object_id', 'true_target']]
tt_b4 = te_def[['object_id']].merge(meta_b4, on='object_id', how='left')['true_target'].to_numpy()
es_conocida = tt_b4 < 990

clf_def = classification.e2gp_al(y_tr_def, random_state=RS).fit(X_tr_def, y_tr_def)
proba_b4 = clf_def.predict_proba(X_te_def)
pred_b4 = clf_def.classes_[proba_b4.argmax(axis=1)]
conf_b4 = proba_b4.max(axis=1)

# Verify numerical agreement before replacing any model or prediction artifact.
reference_predictions = pd.read_csv(rutas_b4.pred_test_csv).set_index('object_id')
reference_confidence = reference_predictions.loc[te_def['object_id'], 'proba_max'].to_numpy()
if not np.allclose(conf_b4, reference_confidence, rtol=1e-8, atol=1e-8):
    raise ValueError('Refitted classifier disagrees with canonical saved confidence. Existing model and predictions were retained.')
print('Refitted model reproduces canonical maximum class probabilities.')
meta_modelo = classification.save_e2gp_artifact(
    clf_def, cols_def, n_train=len(y_tr_def))
print(f"Canonical model: {config.E2GP_MODEL.name} · "
      f"{meta_modelo['n_features']} features · SHA-256 {meta_modelo['model_sha256'][:12]}…")

predicciones_b4 = pd.DataFrame({
    'object_id': te_def['object_id'].to_numpy(),
    'true_target': tt_b4,
    'clase_real': np.where(es_conocida, tt_b4, 99),
    'pred': pred_b4,
    'proba_max': conf_b4,
    'novelty': 1.0 - conf_b4,
    'es_desconocida': ~es_conocida,
})
predicciones_b4.to_csv(rutas_b4.pred_test_csv, index=False)

y_true = tt_b4[es_conocida]; y_pred = pred_b4[es_conocida]
resumen_b4 = pd.DataFrame([{
    'batch': 4, 'variante': 'Calibrated E2-GP (150 trees)',
    'n_features': len(cols_def), 'n_test': len(tt_b4),
    'n_conocidas': int(es_conocida.sum()), 'n_desconocidas': int((~es_conocida).sum()),
    'WF1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    'MF1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    'accuracy': accuracy_score(y_true, y_pred),
    'auc_novelty_99': roc_auc_score((~es_conocida).astype(int), 1.0 - conf_b4),
}])
resumen_b4.to_csv(rutas_b4.test_results_csv, index=False)
display(resumen_b4.round(4))
print(f'guardados: {rutas_b4.pred_test_csv.name} · {rutas_b4.test_results_csv.name}')
